<a href="https://colab.research.google.com/github/laosrb/AI-Final-Project/blob/main/AI_Project_Social_Media.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [3]:
# IMPORT

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer

import lightgbm as lgb
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

# Load Datasets

In [2]:
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/AI Final Project Ryan Bouapheng/Instagram_Analytics.csv'

KeyboardInterrupt: 

In [ ]:
# Load Instagram analytics dataset
instagram_data = pd.read_csv("Instagram_Analytics.csv")

# Load social media engagement dataset
social_engagement = pd.read_csv("Social Media Engagement Dataset.csv")

# Preview data
print(instagram_data.head())
print(social_engagement.head())

# Data Preprocessing

In [1]:
# Example: Clean captions
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.lower()
    text = ''.join([c for c in text if c.isalnum() or c.isspace()])
    return text

instagram_data['clean_caption'] = instagram_data['caption'].apply(clean_text)

# replace missing values
instagram_data.fillna({'likes': 0, 'comments': 0, 'shares': 0}, inplace=True)

# Example: Extract features from posting time
instagram_data['posting_hour'] = pd.to_datetime(instagram_data['posting_time']).dt.hour

# Optional: Encode categorical variables
instagram_data = pd.get_dummies(instagram_data, columns=['post_type'], drop_first=True)

NameError: name 'instagram_data' is not defined

# Feature Engineering

In [ ]:
# TF-IDF for captions
tfidf_vectorizer = TfidfVectorizer(max_features=500)
caption_tfidf = tfidf_vectorizer.fit_transform(instagram_data['clean_caption']).toarray()

# Combine with numeric features
numeric_features = instagram_data[['posting_hour', 'followers_count']].values

X = np.hstack((caption_tfidf, numeric_features))
y = instagram_data['likes'].values  # Target: likes (can also combine engagement metrics)

# Split Training & Testing Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Efficient Model

In [ ]:
# LightGBM model
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

lgb_model.fit(X_train, y_train)
y_pred = lgb_model.predict(X_test)

# Evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R2 Score: {r2}")

# Optimize Algorithms

In [ ]:
def predict_engagement(caption, posting_hour, followers_count):
    caption_clean = clean_text(caption)
    caption_vec = tfidf_vectorizer.transform([caption_clean]).toarray()
    features = np.hstack((caption_vec, np.array([[posting_hour, followers_count]])))
    predicted_likes = lgb_model.predict(features)[0]
    return predicted_likes

# Example usage
caption = "Check out my new summer collection! #fashion #style"
predicted_likes = predict_engagement(caption, posting_hour=15, followers_count=1200)
print(f"Predicted Likes: {int(predicted_likes)}")

# Green AI Strategies for Your Project

## Use Lightweight Models
- Avoid large neural networks.
- Use models like:
  - LightGBM
  - XGBoost
  - Random Forests with limited depth

## Limit Feature Dimensions
- Reduce TF-IDF vector size (e.g., 500–1000 features instead of thousands).
- Drop low-importance numeric and categorical features.

## Early Stopping
- Stop training when performance stops improving to save computation.

## Batch Processing
- Predict in batches instead of per-post to reduce repeated computations.

## Energy Tracking (Optional)
- Use the `time` module to track training duration as a proxy for energy usage.

## Reproducibility
- Fix random seeds to avoid repeated runs for the same results.

# Preprocessing


In [ ]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.lower()
    text = ''.join([c for c in text if c.isalnum() or c.isspace()])
    return text

instagram_data['clean_caption'] = instagram_data['caption'].apply(clean_text)
instagram_data.fillna({'likes': 0, 'comments': 0, 'shares': 0}, inplace=True)

# Extract simple numeric features only (Green AI: minimal complexity)
instagram_data['posting_hour'] = pd.to_datetime(instagram_data['posting_time']).dt.hour


# Feature Engineering (Green AI)

In [ ]:
# TF-IDF limited to 500 features to reduce energy consumption
tfidf_vectorizer = TfidfVectorizer(max_features=500, stop_words=stopwords.words('english'))
caption_tfidf = tfidf_vectorizer.fit_transform(instagram_data['clean_caption']).toarray()

numeric_features = instagram_data[['posting_hour', 'followers_count']].values

X = np.hstack((caption_tfidf, numeric_features))
y = instagram_data['likes'].values


# Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



# Train Lightweight Green AI Model


In [ ]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=200,       # smaller number of trees saves energy
    learning_rate=0.05,
    num_leaves=31,           # limited complexity
    random_state=42
)

# Track training time (proxy for energy consumption)
start_time = time.time()
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='rmse',
    early_stopping_rounds=20,  # stop early to save energy
    verbose=False
)
end_time = time.time()

training_duration = end_time - start_time
print(f"Training Time (seconds): {training_duration:.2f}")



# Model Evaluation


In [ ]:
y_pred = lgb_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R2 Score: {r2:.2f}")



# Prediction Function (Batch-Friendly)


In [ ]:
def predict_engagement(captions, posting_hours, followers_counts):
    # Predict engagement for multiple posts at once (Green AI: batch processing)
    captions_clean = [clean_text(c) for c in captions]
    captions_vec = tfidf_vectorizer.transform(captions_clean).toarray()
    numeric_features = np.array([posting_hours, followers_counts]).T
    features = np.hstack((captions_vec, numeric_features))
    return lgb_model.predict(features)

# Example usage
captions = ["Check out my new summer collection! #fashion #style",
            "Behind the scenes of our latest photoshoot!"]
posting_hours = [15, 18]
followers_counts = [1200, 1500]

predicted_likes = predict_engagement(captions, posting_hours, followers_counts)
print("Predicted Likes:", predicted_likes)

# Enhance Prediction & Scoring Logic
We need to ensure the model produces a score. We'll normalize the predicted likes against your historical maximum to get that (0-100) range.

In [ ]:
import numpy as np

def get_performance_score(predicted_likes, max_historical_likes):
    """Converts raw like prediction to a 0-100 scale."""
    score = (predicted_likes / max_historical_likes) * 100
    return min(100, round(score, 2))

# Calculate this once after loading data
max_likes = instagram_data['likes'].max()

# Optimizer Algorithm
This function doesn't just predict; it iterates through different scenarios (like every hour of the day) to find the "peak" performance for your specific caption.

In [ ]:
def optimize_post(caption, followers_count):
    best_score = -1
    best_hour = 0

    # 1. Analyze the current caption
    # (Extract hashtags if they exist in the string)
    has_hashtags = "#" in caption

    # 2. Iterative Optimization Loop (Testing every hour of the day)
    for hour in range(24):
        # Clean and Transform
        caption_clean = clean_text(caption)
        caption_vec = tfidf_vectorizer.transform([caption_clean]).toarray()
        features = np.hstack((caption_vec, np.array([[hour, followers_count]])))

        # Predict
        prediction = lgb_model.predict(features)[0]
        score = get_performance_score(prediction, max_likes)

        if score > best_score:
            best_score = score
            best_hour = hour

    # 3. Generate Human-Readable Suggestions
    suggestions = []
    if not has_hashtags:
        suggestions.append("Strategy: Adding 3-5 relevant hashtags could increase reach.")

    if len(caption) < 20:
        suggestions.append("Strategy: Your caption is very short. Try adding a Call to Action (CTA).")

    return {
        "Predicted Performance Score": f"{best_score}/100",
        "Recommended Posting Time": f"{best_hour}:00",
        "Optimization Tips": suggestions
    }

# Use Case

In [ ]:
new_post = "Launching my new project today! #tech #innovation"
analysis = optimize_post(new_post, followers_count=1500)

print(f"Post Score: {analysis['Predicted Performance Score']}")
print(f"Best Time to Post: {analysis['Recommended Posting Time']}")
for tip in analysis['Optimization Tips']:
    print(tip)